In [ ]:
import os
os.environ["OPENAI_API_KEY"]="your_openai_api_key_here"  # Replace with your actual OpenAI API key

In [2]:
%pip install -U langchain langchain-openai langchain-community duckduckgo-search

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.4/165.4 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.6/145.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 573.0/573.0 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 57.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.6
    Uninstalling langchain-core-1.6.6:
      Successfully uninstalled langchain-core-1.6.6
  Attempting uninstall: l

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
import requests

In [ ]:
%pip install -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 685.6 kB/s eta 0:00:00


In [3]:
from langchain_community.tools import DuckDuckGoSearchRun
search_tool =DuckDuckGoSearchRun()

/tmp/ipykernel_5734/4021075924.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [ ]:
@tool
def get_weather_data(city: str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  url = f"http://api.weatherstack.com/current?access_key=<YOUR_WEATHERSTACK_API_KEY>&query={city}" #Replace <YOUR_WEATHERSTACK_API_KEY> with your actual API key from the Weatherstack service.
  response = requests.get(url)
  return response.json()

In [5]:
llm = ChatOpenAI(model="gpt-6-luna")

In [6]:
from langchain.agents import create_agent

# CUSTOM PROMPT

In [13]:
agent = create_agent(
    model=llm, #here attribute name is model not llm.
    tools=[search_tool, get_weather_data],
    #CUSTOM -SYSTEM PROMPT
    system_prompt=( #attribute name is system_prompt not prompt
        "You are a helpful assistant. "
        "Use DuckDuckGo search for current or external information. "
        "Use get_weather_data when the user asks for current weather. "
        "Pass the city provided by the user to the weather tool. "
        "Explain the results clearly and accurately."
    ),
    debug=True #detailed execution information. as verbose for agent_executor
)

#and also The new agent uses structured tool calls rather than depending on the legacy ReAct text parser, which also addresses the type of parsing problem you encountered earlier. check in agent executor obj in langchain_agent2 code

In [15]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "find the capital of andhra pradesh, then find its current weather condition"
        }
    ]
}) #the input format is determined by the particular agent API you're using.

# The dictionary key is "messages".
# Its value is a list of messages.
# Each message dictionary contains a "role" and "content" in this example.
# The agent maintains and updates the message state as it runs.

#For the new API, create_agent() uses a state schema that includes messages.

result #ctrl+f "get_weather_data" in [values], [updates]

[values] {'messages': [HumanMessage(content='find the capital of andhra pradesh, then find its current weather condition', additional_kwargs={}, response_metadata={}, id='f5674b53-8145-4cac-90b5-3f320b2be85c')]}
[updates] {'model': {'messages': [AIMessage(content=[{'arguments': '{"query":"official capital of Andhra Pradesh current capital 2025"}', 'call_id': 'call_CEoJ2V6KZ1K1J9L1PfNXHhlR', 'name': 'duckduckgo_search', 'type': 'function_call', 'id': 'fc_09da633e288332d7006acc93657aa087d089093fc901586bbc', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_09da633e288332d7006acc93646f2087d0a5c7f4d30f459938', 'created_at': 1791791972.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_09da633e288332d7006acc93646f2087d0a5c7f4d30f459938', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'official capital of Andhra Pradesh curr

{'messages': [HumanMessage(content='find the capital of andhra pradesh, then find its current weather condition', additional_kwargs={}, response_metadata={}, id='f5674b53-8145-4cac-90b5-3f320b2be85c'),
  AIMessage(content=[{'arguments': '{"query":"official capital of Andhra Pradesh current capital 2025"}', 'call_id': 'call_CEoJ2V6KZ1K1J9L1PfNXHhlR', 'name': 'duckduckgo_search', 'type': 'function_call', 'id': 'fc_09da633e288332d7006acc93657aa087d089093fc901586bbc', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_09da633e288332d7006acc93646f2087d0a5c7f4d30f459938', 'created_at': 1791791972.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_09da633e288332d7006acc93646f2087d0a5c7f4d30f459938', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'official capital of Andhra Pradesh current capital 2025'}, 'id': 'call_CEoJ2V6KZ1

In [16]:
result["messages"]

[HumanMessage(content='find the capital of andhra pradesh, then find its current weather condition', additional_kwargs={}, response_metadata={}, id='f5674b53-8145-4cac-90b5-3f320b2be85c'),
 AIMessage(content=[{'arguments': '{"query":"official capital of Andhra Pradesh current capital 2025"}', 'call_id': 'call_CEoJ2V6KZ1K1J9L1PfNXHhlR', 'name': 'duckduckgo_search', 'type': 'function_call', 'id': 'fc_09da633e288332d7006acc93657aa087d089093fc901586bbc', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_09da633e288332d7006acc93646f2087d0a5c7f4d30f459938', 'created_at': 1791791972.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_09da633e288332d7006acc93646f2087d0a5c7f4d30f459938', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'official capital of Andhra Pradesh current capital 2025'}, 'id': 'call_CEoJ2V6KZ1K1J9L1PfNXHhlR

In [17]:
result['messages'][-1] #last ai message

AIMessage(content=[{'id': 'rs_09da633e288332d7006acc936e1f6087d0bb6ffc0a21378861', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqzJNwMDerjaY_-ySXJsiAw2rQRGFrO2s8fi_gwA9imh3Q_JRHC9Wmylmz6P1-IB6oJExWM4gO37SeG-k5Sm_6H2YQSJ5ql5TeREbaOHKb5oexuRcRlW8EE_btSC9MYgXjC-yAqaBA1rK8U_jDBoNp1ht5CLDcQoWWWS8QSZNt1CZ_PhBaQvqyXX2iRHKrr_AANAhA1uQoW3XDR0EVTMUcCQennpnip7mARNWDXor7pQ5agd5ThXByLydFbJlgoYzWA8MkzcOieARXXXhY5SE2czABm7m7wsGbA_epODr589bpgUtbXAIVryAwcJwohyHpeUDiGxDMFskXPRGR0RKd22P7k6a1r-OuSVJJrEiInxIcuSnhp8K9b4ulr3vJxjFx20ZwA22JkC-sizQu_itgigDa2ylmgzZc6ri_a3Lv61pZrTHKw-VbW0vt9yxBtZb70p9b4gpG5WhLINEbGpcpOcwfvp2lnRacaAd_5mFdYZVdXPHSO7_CY8qEB44WTUdGLcJQaAJGaURv9kSt_mK8_vOM-XLu-vLPBknn0hH3CL_dWeb-fGX9LmsUgPDJHMEKG3Hzd5t2ulxMbG2J1JCP5L9rxhZv3LzZ_3sJ6TwFD0dEhlKN7UQvJISBY_Kr8HXu-NuVAUOZ_LswoTOMTjRi5gY40cJr_b29i-BeX2qqEb5q_bJjAjf_Vb9BmaQSYoCk5jvvYrD43mlkgpKd82RR_A7E9qvlMOrkWV5J5mtlGIGrR3hbqyoXPkUPxOoCQxPwKXEsQlehyHm8ugK0YKM0gTrbPZzAT_EN6IoD0f6PgtJId4hXY_DuY9K4F42Ye_Lxl8tp

In [19]:
result['messages'][-1].text #final

'Andhra Pradesh’s capital is **Amaravati**. The weather lookup matched a different city named Amaravati, so I used nearby **Vijayawada** as a local proxy: it’s currently **34°C with smoky haze**, and feels like **37°C**.'

# BUILT-IN PROMPT
approach not recommended. below code is written just to clarify doubt regarding a way of creating agents in langchain - study the comments so that u will get to know why

In [ ]:
!pip install -U langchain-classic langchainhub

In [8]:
from langchain_classic import hub

In [9]:
from langsmith import Client

client = Client()

prompt = client.pull_prompt(
    "hwchase17/react",
    dangerously_pull_public_prompt=True
)

In [16]:
# agent2 = create_agent(
#     model=llm,
#     tools=[search_tool, get_weather_data],
#     system_prompt= prompt,
#     debug=True
# )

# ERROR cz:
# The error occurs because you're passing the old ReAct PromptTemplate object into the new agent's system_prompt parameter. -> system_prompt=prompt,
# where "prompt" returns PromptTemplate designed for the legacy ReAct agent. However, create_agent() expects system_prompt to be a string (or a supported system message), not that legacy template.

In [11]:
agent2 = create_agent(
    model=llm,
    tools=[search_tool, get_weather_data],
    system_prompt= prompt.template, #fix 1 : give a custom prompt , fix 2: you could extract its template text
    debug=True
)
#But I don't recommend this for your current agent. The old hwchase17/react template contains placeholders such as {tools}, {tool_names}, and {agent_scratchpad}.
#These belong to the legacy ReAct prompting mechanism and won't automatically be populated by create_agent().

In [14]:
result2 = agent2.invoke({"messages": [
        {
            "role": "user",
            "content": "find the capital of andhra pradesh, then find its current weather condition"
        }
    ]})
result2

[values] {'messages': [HumanMessage(content='find the capital of andhra pradesh, then find its current weather condition', additional_kwargs={}, response_metadata={}, id='df114cf3-321c-43fb-bf77-0a755159ded9')]}
[updates] {'model': {'messages': [AIMessage(content=[{'id': 'rs_0a6521dccb88d884006acc974b358487d0af19250d0096bc53', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqzJdNEeFZCriYALEmeXL9XBmczMaVoR4tD0T8td2YoUiig3xWvjeg4jwokylirxA-Iu8kizPPfTaFSUrDNAoZpp362Gsbf5ojY2BZe15vV9R-iRh6CiIomsi_M561lE_e0sfrUWcjDzkOK9ZYTHisb-GIjor7Yuv2S8H3ySYgoYzitb5UAVoD2iVymebeaDkRIF8Hk0PjzZ7byF3zi_r-4w91JaferWg7iCiX53yNpWU4RFAz5p8CeBBDwrg1o8ar48J8x-NVChROYtP1VdTjXEBFLOIbhgxsCxU6YLnki9ila3ZH7G_TMM5kodMPBD9f_Ovj6OB1sbgT7CQiwFhpF-anqxNGSq3Tvqr_YYtlDnB8XUsisg51eg4CJM0nFqs5ntbZmtBiiSzRP3cSganfK4X0jjKlfywEJw3BjdVQG4yxnj4EF6tTTCp8skIzIQ180JbU_3taHSGPmQXTMeOikiGgBhMTC8SZQ2Ck1xLp-yLcNVTCswOKrcn5VD_EdHsoPPYANEPhhQ2MEvC12oxbcDYeFZXyX3roExdCWSsV5_K2SGEI6EUdO2b7K_Mhc0FhPVb56AYWWVzf_Q

{'messages': [HumanMessage(content='find the capital of andhra pradesh, then find its current weather condition', additional_kwargs={}, response_metadata={}, id='df114cf3-321c-43fb-bf77-0a755159ded9'),
  AIMessage(content=[{'id': 'rs_0a6521dccb88d884006acc974b358487d0af19250d0096bc53', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqzJdNEeFZCriYALEmeXL9XBmczMaVoR4tD0T8td2YoUiig3xWvjeg4jwokylirxA-Iu8kizPPfTaFSUrDNAoZpp362Gsbf5ojY2BZe15vV9R-iRh6CiIomsi_M561lE_e0sfrUWcjDzkOK9ZYTHisb-GIjor7Yuv2S8H3ySYgoYzitb5UAVoD2iVymebeaDkRIF8Hk0PjzZ7byF3zi_r-4w91JaferWg7iCiX53yNpWU4RFAz5p8CeBBDwrg1o8ar48J8x-NVChROYtP1VdTjXEBFLOIbhgxsCxU6YLnki9ila3ZH7G_TMM5kodMPBD9f_Ovj6OB1sbgT7CQiwFhpF-anqxNGSq3Tvqr_YYtlDnB8XUsisg51eg4CJM0nFqs5ntbZmtBiiSzRP3cSganfK4X0jjKlfywEJw3BjdVQG4yxnj4EF6tTTCp8skIzIQ180JbU_3taHSGPmQXTMeOikiGgBhMTC8SZQ2Ck1xLp-yLcNVTCswOKrcn5VD_EdHsoPPYANEPhhQ2MEvC12oxbcDYeFZXyX3roExdCWSsV5_K2SGEI6EUdO2b7K_Mhc0FhPVb56AYWWVzf_QgC6X6XUwFE4zuAAATe0PlaGb8Q6uN0rQXs5fYLqm2-

In [15]:
result2['messages'][-1].text

#Yes, it used the template text as the system prompt. You did provide a prompt; you just didn't write a new one yourself.
# However, prompt.template extracts the text from your old hwchase17/react prompt. It does not automatically adapt that text to the new create_agent() API.

#Your old ReAct prompt contains placeholders such as:
# {tools}
# {tool_names}
# {agent_scratchpad}


# In the legacy API, these placeholders are populated as part of the ReAct prompt-formatting process. In the new API, passing prompt.template as system_prompt
# does not automatically fill those placeholders.
# Your model could still call the tools because create_agent() registered them through tools=[search_tool, get_weather_data]. The tool definitions are provided separately to the model.
# So, your successful result does not prove that the legacy prompt was being used correctly.

'**Capital:** Amaravati  \n**Current weather in Amaravati:** Smoky haze, 33°C (feels like 37°C), as of 1:46 p.m. local time.'